<a href="https://colab.research.google.com/github/justorfc/Estadistica_Aplicada_con_Python_y_R/blob/main/2_Semana_2_An%C3%A1lisis_Exploratorio_de_Datos_(EDA)_y_Visualizaciones_Cruzadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este Notebook contiene la estructura detallada para la **Semana 2**, enfocada en el Análisis Exploratorio de Datos (EDA) y visualizaciones cruzadas. El material está contextualizado para los sistemas de producción, evaluando cómo interactúan el clima y el suelo para determinar el rendimiento agrícola.

A continuación, se presenta el contenido estructurado que puedes copiar directamente en las celdas de tu cuaderno de Google Colab.

---

# Semana 2: Análisis Exploratorio de Datos (EDA) y Visualizaciones Cruzadas
**Asignatura:** Estadística Aplicada con Python y R  
**Programa:** Ingeniería Agrícola - Universidad de Sucre  
**Profesor:** Justo Rafael Fuentes Cuello  

---

### Situación de Interés: Descubriendo patrones en el sistema suelo-clima-cultivo
En la ingeniería agrícola, rara vez una variable actúa sola. El rendimiento de un cultivo depende de la interacción compleja entre la humedad del suelo, la precipitación y la evapotranspiración.

Hoy aplicaremos **Análisis Exploratorio de Datos (EDA)** para entender el comportamiento individual de estas variables (análisis univariado) y cómo se relacionan entre sí (análisis bivariado y multivariado). Nuestro objetivo es buscar patrones, identificar correlaciones y distinguir la variabilidad natural de posibles valores atípicos.

In [ ]:
# Importamos las librerías fundamentales para manipulación de datos y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuramos el estilo visual de los gráficos
sns.set_theme(style="whitegrid")

# Generamos un conjunto de datos simulado de 150 parcelas agrícolas
np.random.seed(42) # Semilla para reproducibilidad

# Simulamos variables agroclimáticas y de producción
precipitacion = np.random.normal(loc=120, scale=30, size=150) # Precipitación mensual (mm)
evapotranspiracion = np.random.normal(loc=90, scale=15, size=150) # ETo (mm)

# La humedad del suelo estará correlacionada con la precipitación
humedad_suelo = precipitacion * 0.2 + np.random.normal(loc=10, scale=5, size=150)

# El rendimiento (ton/ha) dependerá de la humedad, penalizando excesos y déficits
rendimiento = 10 - 0.05 * (humedad_suelo - 35)**2 + np.random.normal(loc=0, scale=1.5, size=150)
rendimiento = np.clip(rendimiento, 1, 15) # Limitamos a valores agronómicamente lógicos

# Creamos el DataFrame
df_agricola = pd.DataFrame({
    'Precipitacion_mm': precipitacion,
    'Evapotranspiracion_mm': evapotranspiracion,
    'Humedad_Suelo_%': humedad_suelo,
    'Rendimiento_ton_ha': rendimiento
})

print("Dataset agrícola simulado generado con éxito. Dimensión:", df_agricola.shape)
df_agricola.head()

### 1. Estadística Descriptiva (Medidas de Tendencia y Dispersión)
Antes de graficar, debemos comprender la magnitud de nuestros datos. ¿Cuál es el rendimiento promedio? ¿Qué tan variable es la lluvia en nuestras parcelas?

In [ ]:
# Generamos un resumen estadístico de todas las variables numéricas
resumen = df_agricola.describe()
resumen = round(resumen, 2) # Redondeamos a dos decimales para facilitar la lectura
resumen

### 2. Análisis Univariado: La distribución del Rendimiento

Vamos a analizar la variable objetivo (`Rendimiento_ton_ha`). Utilizaremos un histograma combinado con una curva de densidad (KDE) para ver la forma de la distribución, y un diagrama de caja (Boxplot) para identificar cuartiles y posibles atípicos.

In [ ]:
# Configuramos un lienzo con dos subgráficos (1 fila, 2 columnas)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma y curva de densidad
sns.histplot(df_agricola['Rendimiento_ton_ha'], kde=True, color='green', ax=axes[0])
axes[0].set_title('Distribución del Rendimiento Agrícola')
axes[0].set_xlabel('Rendimiento (ton/ha)')
axes[0].set_ylabel('Frecuencia')

# Diagrama de caja (Boxplot)
sns.boxplot(x=df_agricola['Rendimiento_ton_ha'], color='lightgreen', ax=axes[1])
axes[1].set_title('Boxplot del Rendimiento')
axes[1].set_xlabel('Rendimiento (ton/ha)')

plt.tight_layout()
plt.show()

### 3. Visualizaciones Cruzadas (Análisis Bivariado)
¿Cómo afecta la humedad del suelo al rendimiento final? Un diagrama de dispersión (Scatterplot) es la mejor herramienta para visualizar la relación entre dos variables continuas.

In [ ]:
plt.figure(figsize=(8, 6))

# Diagrama de dispersión: Humedad vs Rendimiento
# Usamos el color (hue) para mostrar un tercer factor: la precipitación
scatter = sns.scatterplot(
    data=df_agricola,
    x='Humedad_Suelo_%',
    y='Rendimiento_ton_ha',
    hue='Precipitacion_mm',
    palette='viridis',
    s=70,
    alpha=0.8
)

plt.title('Impacto de la Humedad del Suelo en el Rendimiento')
plt.xlabel('Humedad Volumétrica del Suelo (%)')
plt.ylabel('Rendimiento (ton/ha)')
plt.legend(title='Precipitación (mm)')

plt.show()

### 4. La Fotografía Completa: Matriz de Correlación y Pairplot

Para entender el sistema completo, generaremos un gráfico de pares (`pairplot`) que cruza todas las variables contra todas. Finalmente, calcularemos la correlación de Pearson para cuantificar matemáticamente la fuerza de estas relaciones.

In [ ]:
# Gráfico de relaciones múltiples
sns.pairplot(df_agricola, diag_kind='kde', corner=True, plot_kws={'alpha': 0.6})
plt.suptitle('Matriz de Dispersión de Variables Agroclimáticas', y=1.02)
plt.show()

# Matriz de Correlación Matemática (Heatmap)
plt.figure(figsize=(6, 5))
matriz_corr = df_agricola.corr()

# Dibujamos un mapa de calor con los coeficientes de Pearson
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title('Matriz de Correlación de Pearson')
plt.show()

### 🛑 Reflexión y Reserva Cognitiva (Discusión y síntesis manuscrita)

Tomando tu cuaderno de apuntes físico, responde:
1. Observa el diagrama de dispersión (Humedad vs. Rendimiento). ¿La relación es estrictamente lineal? ¿Qué le ocurre agronómicamente al cultivo si la humedad es muy alta o muy baja?
2. Observa el mapa de calor (Matriz de correlación). ¿Cuál es la variable que más se correlaciona con la Humedad del Suelo? ¿Tiene sentido físico este comportamiento?
3. ¿Por qué es peligroso asumir que una alta correlación (número cercano a 1 o -1) implica necesariamente una relación de "causa y efecto"?

---

### Instrucciones para el reto en R (Trabajo Autónomo y Sesión 2)

**Misión:** Has descubierto patrones vitales usando Python, `pandas` y `seaborn`. Ahora tu objetivo es realizar exactamente este mismo flujo exploratorio en **R**, aprovechando la potencia gráfica de **`ggplot2`** dentro de **Posit Cloud**.

**Pasos a seguir:**
1. Ingresa a tu espacio de trabajo en Posit Cloud y abre un nuevo documento **RMarkdown**.
2. Necesitarás consultar a tu asistente de IA (ChatGPT, Gemini, Claude, etc.) para que te ayude a traducir la lógica.
3. Utiliza el siguiente *prompt* de inicio para guiar al asistente:
   > *"Actúa como un profesor de Estadística y R. Acabo de realizar un Análisis Exploratorio de Datos (EDA) agrícola en Python usando pandas y seaborn. Generé histogramas, boxplots, gráficos de dispersión y mapas de correlación. Necesito replicar este mismo EDA en R usando `tidyverse`, `ggplot2` y `corrplot` (o similar). Escribe el código en R para generar datos simulados similares (Precipitación, Evapotranspiración, Humedad y Rendimiento) y luego crea las visualizaciones (histograma, boxplot, dispersión y correlación). Explícame cómo estructurar esto en chunks de código dentro de RMarkdown."*
4. Examina la sintaxis propuesta. Notarás que `ggplot2` funciona por "capas" agregando elementos con el signo `+`.
5. Ejecuta cada bloque de código y compara visualmente los gráficos obtenidos en R con los de Python.
6. **Entrega final:** Renderiza tu RMarkdown a formato HTML o PDF. Al final del documento, incluye una sección de **"Bitácora de IA"** detallando cómo interactuaste con el chatbot, qué ajustes hiciste al código de R para mejorarlo y tus conclusiones sobre cuál librería gráfica te pareció más intuitiva.